# Validation Checks

This notebook validates the final experiment outputs by checking output completeness, image readability, image dimensions, metadata availability and metric file consistency.


In [14]:
from pathlib import Path
from PIL import Image
from IPython.display import display
import pandas as pd

repo_dir = Path("/content/image-data-generation")
results_dir = repo_dir / "results"
validation_dir = results_dir / "validation"

validation_dir.mkdir(parents=True, exist_ok=True)

EXPECTED_SIZE = (512, 512)

In [15]:
# Expected output structure for each experiemnt
experiments = {
    "baseline_vs_controlnet": {
        "path": results_dir / "baseline_vs_controlnet",
        "structure": "flat",
        "expected_runs": 15,
        "required_files": [
            "original.png",
            "baseline_sd.png",
            "controlnet_depth.png",
            "comparison.png",
            "metadata.txt"
        ],
        "image_size_check_files": [
            "original.png",
            "baseline_sd.png",
            "controlnet_depth.png"
        ]
    },

    "controlnet_vs_ipadapter": {
        "path": results_dir / "controlnet_vs_ipadapter",
        "structure": "flat",
        "expected_runs": 15,
        "required_files": [
            "original.png",
            "reference_condition.png",
            "controlnet_only.png",
            "controlnet_ipadapter.png",
            "comparison.png",
            "metadata.txt"
        ],
        "image_size_check_files": [
            "original.png",
            "reference_condition.png",
            "controlnet_only.png",
            "controlnet_ipadapter.png"
        ]
    },

    "full_pipeline_ablation": {
        "path": results_dir / "full_pipeline_ablation",
        "structure": "flat",
        "expected_runs": 15,
        "required_files": [
            "original.png",
            "reference_condition.png",
            "baseline_sd.png",
            "without_controlnet.png",
            "without_ipadapter.png",
            "without_blip2.png",
            "full_pipeline.png",
            "comparison.png",
            "metadata.txt"
        ],
        "image_size_check_files": [
            "original.png",
            "reference_condition.png",
            "baseline_sd.png",
            "without_controlnet.png",
            "without_ipadapter.png",
            "without_blip2.png",
            "full_pipeline.png"
        ]
    },

    "multi_weather_evaluation": {
        "path": results_dir / "multi_weather_evaluation",
        "structure": "nested",
        "expected_runs": 90,
        "required_files": [
            "original.png",
            "reference_condition.png",
            "generated_output.png",
            "comparison.png",
            "metadata.txt"
        ],
        "image_size_check_files": [
            "original.png",
            "reference_condition.png",
            "generated_output.png"
        ]
    }
}

In [16]:
# Verify images can be opened, RGB format and valid size
def check_image_file(path, expected_size=(512, 512)):
    try:
        img = Image.open(path).convert("RGB")
        return {
            "readable": True,
            "image_size": str(img.size),
            "image_mode": img.mode,
            "size_correct": img.size == expected_size,
            "error": ""
        }
    except Exception as e:
        return {
            "readable": False,
            "image_size": None,
            "image_mode": None,
            "size_correct": False,
            "error": str(e)
        }

# Returns folders that should be validated
def get_run_dirs(experiment_path, structure):
    if not experiment_path.exists():
        return []

    if structure == "flat":
        return sorted([p for p in experiment_path.iterdir() if p.is_dir()])

    if structure == "nested":
        run_dirs = []
        road_dirs = sorted([p for p in experiment_path.iterdir() if p.is_dir()])

        for road_dir in road_dirs:
            weather_dirs = sorted([p for p in road_dir.iterdir() if p.is_dir()])
            run_dirs.extend(weather_dirs)

        return run_dirs

    return []

In [17]:
# Validate experiemnts
validation_rows = []

for experiment_name, config in experiments.items():
    experiment_path = config["path"]
    structure = config["structure"]
    required_files = config["required_files"]
    image_size_check_files = config["image_size_check_files"]

    if not experiment_path.exists():
        validation_rows.append({
            "experiment": experiment_name,
            "run_folder": "EXPERIMENT_FOLDER",
            "file": str(experiment_path),
            "exists": False,
            "readable": None,
            "image_size": None,
            "image_mode": None,
            "size_correct": None,
            "error": "Experiment folder missing"
        })
        continue

    run_dirs = get_run_dirs(experiment_path, structure)

    for run_dir in run_dirs:
        for file_name in required_files:
            file_path = run_dir / file_name
            exists = file_path.exists()

            row = {
                "experiment": experiment_name,
                "run_folder": str(run_dir.relative_to(results_dir)),
                "file": file_name,
                "exists": exists,
                "readable": None,
                "image_size": None,
                "image_mode": None,
                "size_correct": None,
                "error": ""
            }

            if exists and file_name in image_size_check_files:
                image_check = check_image_file(file_path, EXPECTED_SIZE)
                row.update(image_check)

            validation_rows.append(row)

validation_df = pd.DataFrame(validation_rows)

validation_summary_path = validation_dir / "output_validation_summary.csv"
validation_df.to_csv(validation_summary_path, index=False)

validation_df.head()

,experiment,run_folder,file,exists,readable,image_size,image_mode,size_correct,error
0,baseline_vs_controlnet,baseline_vs_controlnet/road_01,original.png,True,True,"(512, 512)",RGB,True,
1,baseline_vs_controlnet,baseline_vs_controlnet/road_01,baseline_sd.png,True,True,"(512, 512)",RGB,True,
2,baseline_vs_controlnet,baseline_vs_controlnet/road_01,controlnet_depth.png,True,True,"(512, 512)",RGB,True,
3,baseline_vs_controlnet,baseline_vs_controlnet/road_01,comparison.png,True,None,None,None,None,
4,baseline_vs_controlnet,baseline_vs_controlnet/road_01,metadata.txt,True,None,None,None,None,


In [18]:
# Check run counts
run_count_rows = []

for experiment_name, config in experiments.items():
    experiment_path = config["path"]
    run_dirs = get_run_dirs(experiment_path, config["structure"])

    run_count_rows.append({
        "experiment": experiment_name,
        "expected_runs": config["expected_runs"],
        "actual_runs": len(run_dirs),
        "count_correct": len(run_dirs) == config["expected_runs"]
    })

run_count_df = pd.DataFrame(run_count_rows)

run_count_path = validation_dir / "run_count_validation.csv"
run_count_df.to_csv(run_count_path, index=False)

run_count_df

,experiment,expected_runs,actual_runs,count_correct
0,baseline_vs_controlnet,15,15,True
1,controlnet_vs_ipadapter,15,15,True
2,full_pipeline_ablation,15,15,True
3,multi_weather_evaluation,90,90,True


In [19]:
# Check failures
failures = validation_df[
    (validation_df["exists"] == False) |
    (validation_df["readable"] == False) |
    (validation_df["size_correct"] == False)
].copy()

failures_path = validation_dir / "output_validation_failures.csv"
failures.to_csv(failures_path, index=False)

print("Total validation rows:", len(validation_df))
print("Total output failures:", len(failures))

if len(failures) > 0:
    display(failures.head(30))
else:
    print("All expected output files exist, checked images are readable, and checked image sizes are 512x512.")

Total validation rows: 750
Total output failures: 0
All expected output files exist, checked images are readable, and checked image sizes are 512x512.


In [20]:
# Validate file existence
metrics_dir = results_dir / "metrics"

required_metric_files = [
    "metrics_summary.csv",
    "metrics_average_summary.csv",
    "metrics_report_table.csv",
    "ssim_summary.png",
    "lpips_summary.png",
    "clip_score_summary.png",
    "runtime_summary.png"
]

metric_file_rows = []

for file_name in required_metric_files:
    file_path = metrics_dir / file_name

    metric_file_rows.append({
        "file": file_name,
        "exists": file_path.exists(),
        "path": str(file_path)
    })

metric_files_df = pd.DataFrame(metric_file_rows)

metric_files_path = validation_dir / "metric_file_validation.csv"
metric_files_df.to_csv(metric_files_path, index=False)

metric_files_df

,file,exists,path
0,metrics_summary.csv,True,/content/image-data-generation/results/metrics...
1,metrics_average_summary.csv,True,/content/image-data-generation/results/metrics...
2,metrics_report_table.csv,True,/content/image-data-generation/results/metrics...
3,ssim_summary.png,True,/content/image-data-generation/results/metrics...
4,lpips_summary.png,True,/content/image-data-generation/results/metrics...
5,clip_score_summary.png,True,/content/image-data-generation/results/metrics...
6,runtime_summary.png,True,/content/image-data-generation/results/metrics...


In [21]:
# Validate CSV file content
csv_validation_rows = []

expected_csv_columns = {
    "metrics_summary.csv": [
        "experiment",
        "image_id",
        "model",
        "ssim",
        "lpips",
        "clip_score",
        "runtime_seconds"
    ],
    "metrics_average_summary.csv": [
        "experiment",
        "model",
        "weather_condition",
        "ssim",
        "lpips",
        "clip_score",
        "runtime_seconds",
        "label"
    ],
    "metrics_report_table.csv": [
        "Model / Configuration",
        "SSIM",
        "LPIPS",
        "CLIP Score",
        "Runtime (s)"
    ]
}

for csv_name, expected_columns in expected_csv_columns.items():
    csv_path = metrics_dir / csv_name

    if not csv_path.exists():
        csv_validation_rows.append({
            "csv_file": csv_name,
            "exists": False,
            "rows": None,
            "missing_metric_values": None,
            "columns": None,
            "expected_columns_present": False
        })
        continue

    df = pd.read_csv(csv_path)

    expected_columns_present = all(col in df.columns for col in expected_columns)

    # Check missing values
    if csv_name == "metrics_report_table.csv":
        metric_columns = ["SSIM", "LPIPS", "CLIP Score", "Runtime (s)"]
    else:
        metric_columns = ["ssim", "lpips", "clip_score", "runtime_seconds"]

    available_metric_columns = [
        col for col in metric_columns
        if col in df.columns
    ]

    missing_metric_values = int(df[available_metric_columns].isna().sum().sum())

    csv_validation_rows.append({
        "csv_file": csv_name,
        "exists": True,
        "rows": len(df),
        "missing_metric_values": missing_metric_values,
        "columns": ", ".join(df.columns),
        "expected_columns_present": expected_columns_present
    })

csv_validation_df = pd.DataFrame(csv_validation_rows)

csv_validation_path = validation_dir / "metric_csv_validation.csv"
csv_validation_df.to_csv(csv_validation_path, index=False)

csv_validation_df

,csv_file,exists,rows,missing_metric_values,columns,expected_columns_present
0,metrics_summary.csv,True,225,0,"experiment, image_id, model, weather_condition...",True
1,metrics_average_summary.csv,True,15,0,"experiment, model, weather_condition, referenc...",True
2,metrics_report_table.csv,True,15,0,"Model / Configuration, SSIM, LPIPS, CLIP Score...",True


In [22]:
output_checks_passed = len(failures) == 0
run_counts_passed = run_count_df["count_correct"].all()
metric_files_passed = metric_files_df["exists"].all()
metric_csvs_passed = (csv_validation_df["exists"].all() and csv_validation_df["expected_columns_present"].all() and (csv_validation_df["missing_metric_values"] == 0).all())
overall_passed = (output_checks_passed and run_counts_passed and metric_files_passed and metric_csvs_passed)

overall_result_path = validation_dir / "overall_validation_result.txt"

with open(overall_result_path, "w") as f:
    f.write(f"Overall validation passed: {overall_passed}\n")
    f.write(f"Output checks passed: {output_checks_passed}\n")
    f.write(f"Run counts passed: {run_counts_passed}\n")
    f.write(f"Metric files passed: {metric_files_passed}\n")
    f.write(f"Metric CSVs passed: {metric_csvs_passed}\n")
    f.write(f"Output failures: {len(failures)}\n")

print("Overall validation passed:", overall_passed)
print("Saved validation files to:", validation_dir)

Overall validation passed: True
Saved validation files to: /content/image-data-generation/results/validation
